In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

In [ ]:
# Task 1: Write your code here: - Read the dataset Q1_data.csv using read_csv()
path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(path)

In [ ]:
# Task 2: Write your code here: - Inspect the first few rows using head()
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Task 3: Write your code here: - Display dataset information using info()
df.info()

In [ ]:
# Task 4: Write your code here: - Show statistical description using describe()
df.describe()

In [ ]:
# Task 5: Write your code here: - Plot the target distribution (delivery_time)
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=30, edgecolor='black')
plt.title('Delivery_Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here: - Drop the 'Order_ID' column from the data
df = df.drop(columns="Order_ID", axis=1)
df.head()

In [ ]:
# Task 2: Write your code here: - Handle missing values appropriately (Hint: I guess you want to have a closer look at the columns with missing values :) )
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
# Define stat columns
stat_cols = ['Weather', 'Traffic_Level', 'Time_of_Day']

# Drop rows with missing stat values
df = df.dropna(subset=stat_cols).copy()
print(f"Shape after cleaning: {df.shape}")

In [ ]:
print("Missing values:")
print(df.isnull().sum())

In [ ]:
df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].fillna('unknown')
print(df.isnull().sum())

In [ ]:
df['Delivery_Time'] = df['Delivery_Time'].fillna('unknown')
print(df.isnull().sum())

In [ ]:
print(f"Shape after cleaning: {df.shape}")

In [ ]:
# Task 3: Write your code here: - Check and remove duplicates if any exist
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here: - Encode categorical variables if needed (Bonus if used One Hot Encoding)
categorical_cols = df.select_dtypes(include=["object"]).columns
print("Categorical Columns:", list(categorical_cols))

In [ ]:
from sklearn.preprocessing import LabelEncoder

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

# y = df['Delivery_Time']
# y = le.fit_transform(y)

df.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 6: Write your code here: - Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()
  plt.show()

check_target_imbalance(df, "Delivery_Time") # it seems that theres a target imbalance

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split, KFold, cross_val_score

In [ ]:
# Task 1: Write your code here:
feature_cols = df.drop("Delivery_Time", axis=1).astype(float)

X = feature_cols
y = df['Delivery_Time'].astype(float)

In [ ]:
# Show full dataset class distribution
full_ratio = (y.value_counts(normalize=True) * 100).sort_index()
print("Full Dataset Class Distribution")
print("  y class percentages:", {k: f"{v:.2f}%" for k, v in full_ratio.items()})
print("-" * 40)

In [ ]:
# Task 2,3,4,5: Write your code here:

# Stratified split
# Define K-Fold Cross Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Iterate through folds
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    print(f"Fold {fold}")
    print("  X_train shape:", X_train.shape)
    print("  X_test shape :", X_test.shape)
    print("  y_train shape:", y_train.shape)
    print("  y_test shape :", y_test.shape)

    # showing class distribution
    train_ratio = (y_train.value_counts(normalize=True) * 100).sort_index()
    test_ratio = (y_test.value_counts(normalize=True) * 100).sort_index()

    print("  y_train class percentages:", {k: f"{v:.2f}%" for k, v in train_ratio.items()})
    print("  y_test class percentages :", {k: f"{v:.2f}%" for k, v in test_ratio.items()})

    print("-" * 40)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train RandomForestClassifier
model = RandomForestClassifier(n_estimators=100, max_depth=15,
                               class_weight='balanced', random_state=42)
model.fit(X_train, y_train)
print("Model trained!")

In [ ]:
from sklearn.metrics import mean_absolute_error

In [ ]:
# Predict and evaluate
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)

print(f"MAE:  ${mae:,.2f}")

In [ ]:
# Task 1: Write your code here:
# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: